# importing required libraries

In [1]:
#importing pandas for data manipulation and reading csv files
import pandas as pd

#importing numpy for numerical operations
import numpy as np

#importing tools from scikit-learn for splitting data and building the model
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

#importing evaluation metrics to check model accuracy
from sklearn.metrics import mean_absolute_error, mean_squared_error

#importing joblib to save the trained model later
import joblib

import os
from prophet import Prophet

# loading the datasets (sales and events)!

In [3]:
# defining the file paths based on our folder structure
sales_path = '../data/Sales.csv'
events_path = '../data/Events.csv'

# loading the sales data into a pandas dataframe
sales_df = pd.read_csv(sales_path)

# loading the events data into a pandas dataframe
events_df = pd.read_csv(events_path)

In [4]:
# displaying the first few rows of the sales dataset to confirm it loaded correctly
print("sales data preview:")
display(sales_df.head())

sales data preview:


,order_id,product_id,region,sale_date,quantity,discount
0,O001,P001,Hyderabad,2026-02-01,2,10
1,O002,P003,Delhi,2026-02-02,5,0
2,O003,P004,Mumbai,2026-02-02,1,5
3,O004,P007,Chennai,2026-02-03,2,15
4,O005,P010,Mumbai,2026-02-04,3,0


In [5]:
# displaying the first few rows of the events dataset to confirm it loaded correctly
print("\nevents data preview:")
display(events_df.head())


events data preview:


,event_name,sport,start_date,end_date,region
0,IPL 2026,Cricket,2026-03-01,2026-05-30,India
1,Football World Cup,Football,2026-06-01,2026-07-15,Global
2,Wimbledon,Badminton,2026-07-01,2026-07-14,Global
3,Olympics,Gym,2026-08-01,2026-08-20,Global
4,India vs England Series,Cricket,2026-02-15,2026-03-15,India


In [6]:
sales_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   order_id    25 non-null     object
 1   product_id  25 non-null     object
 2   region      25 non-null     object
 3   sale_date   25 non-null     object
 4   quantity    25 non-null     int64 
 5   discount    25 non-null     int64 
dtypes: int64(2), object(4)
memory usage: 1.3+ KB


In [7]:
events_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   event_name  7 non-null      object
 1   sport       7 non-null      object
 2   start_date  7 non-null      object
 3   end_date    7 non-null      object
 4   region      7 non-null      object
dtypes: object(5)
memory usage: 408.0+ bytes


# data preprocessing and merging

In [8]:
# loading products data because we need the 'sport' column to match sales with events
products_path = '../data/Products.csv'
products_df = pd.read_csv(products_path)

In [9]:
# merging sales with products to bring in the 'sport' and 'category' for each sale
# using a left join to ensure we keep all our sales records
merged_df = pd.merge(sales_df, products_df[['product_id', 'sport', 'category']], on='product_id', how='left')

In [10]:
# converting date columns into pandas datetime objects so we can do time-based math
merged_df['sale_date'] = pd.to_datetime(merged_df['sale_date'])
events_df['start_date'] = pd.to_datetime(events_df['start_date'])
events_df['end_date'] = pd.to_datetime(events_df['end_date'])

In [11]:
# displaying the new merged dataset to verify the 'sport' column is added
print("merged sales and products data:")
display(merged_df.head())

merged sales and products data:


,order_id,product_id,region,sale_date,quantity,discount,sport,category
0,O001,P001,Hyderabad,2026-02-01,2,10,Cricket,Bat
1,O002,P003,Delhi,2026-02-02,5,0,Cricket,Balls
2,O003,P004,Mumbai,2026-02-02,1,5,Football,Shoes
3,O004,P007,Chennai,2026-02-03,2,15,Badminton,Racket
4,O005,P010,Mumbai,2026-02-04,3,0,Gym,Gym Equipment


# feature engineering (dates and event flags)

In [12]:
# extracting time-based features from the sale date
merged_df['sale_month'] = merged_df['sale_date'].dt.month
merged_df['sale_day'] = merged_df['sale_date'].dt.day
merged_df['sale_dayofweek'] = merged_df['sale_date'].dt.dayofweek

In [13]:
# mapping cities to their broader event regions so our event logic works
city_to_country_map = {
    'Hyderabad': 'India',
    'Delhi': 'India',
    'Mumbai': 'India',
    'Chennai': 'India',
    'Bangalore': 'India',
    'London': 'UK'
}

In [14]:
# adding a 'country' column based on the region (city)
merged_df['country'] = merged_df['region'].map(city_to_country_map)

In [15]:
# creating a function to check if a sale happened during an active event
def is_event_active(row):
    # finding events in our events dataframe that match the sport of the current sale
    matching_events = events_df[events_df['sport'] == row['sport']]
    
    for _, event in matching_events.iterrows():
        # checking if the sale date falls between the event start and end dates
        date_match = event['start_date'] <= row['sale_date'] <= event['end_date']
        
        # checking if the event region matches the sale's country, or if it is a 'global' event
        region_match = (event['region'] == 'Global') or (event['region'] == row['country'])
        
        # if both the date and region match, we flag it as an active event (1)
        if date_match and region_match:
            return 1 
            
    # if no events match, return 0
    return 0 

In [16]:
# applying our function row by row to create the new 'event_active' column
merged_df['event_active'] = merged_df.apply(is_event_active, axis=1)

# displaying a few columns to verify our event flag works properly
print("data with engineered features (checking event flag):")
display(merged_df[['sale_date', 'sport', 'region', 'country', 'event_active', 'quantity']].head(10))

data with engineered features (checking event flag):


,sale_date,sport,region,country,event_active,quantity
0,2026-02-01,Cricket,Hyderabad,India,0,2
1,2026-02-02,Cricket,Delhi,India,0,5
2,2026-02-02,Football,Mumbai,India,0,1
3,2026-02-03,Badminton,Chennai,India,0,2
4,2026-02-04,Gym,Mumbai,India,0,3
5,2026-02-05,Cricket,Hyderabad,India,0,4
6,2026-02-05,Cricket,Delhi,India,0,1
7,2026-02-06,Football,London,UK,0,2
8,2026-02-07,Badminton,Bangalore,India,0,1
9,2026-02-07,Gym,Mumbai,India,0,1


# preparing data for training (features and target)

In [17]:
# selecting the features (inputs) for our machine learning model
# we use the discount and the time/event features we just created
features = ['discount', 'sale_month', 'sale_day', 'sale_dayofweek', 'event_active']
x = merged_df[features]

# selecting the target variable (what we want to predict)
# we want to predict the quantity of items sold
y = merged_df['quantity']

In [18]:
# splitting the data into training and testing sets (80% train, 20% test)
# setting a random state ensures we get the same split every time we run the code
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

# displaying the shape of our training and testing data to confirm the split
print("training data shape (features, target):", x_train.shape, y_train.shape)
print("testing data shape (features, target):", x_test.shape, y_test.shape)

training data shape (features, target): (20, 5) (20,)
testing data shape (features, target): (5, 5) (5,)


# training the machine learning model (random forest)

In [19]:
# initializing the random forest regressor model
# we use 100 trees (n_estimators) and set a random state for reproducibility
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)

# training the model using our training data
# the model learns the relationship between our features (like events and discounts) and the quantity sold
rf_model.fit(x_train, y_train)

# printing a success message to know it finished
print("model training completed successfully!")

model training completed successfully!


In [20]:
!pip install prophet -q

# training an alternative time-series model (prophet)

In [21]:
# creating a separate dataframe just for prophet using the sale date and quantity
prophet_df = merged_df[['sale_date', 'quantity']].copy()

# renaming the columns because prophet strictly requires 'ds' for dates and 'y' for the target
prophet_df = prophet_df.rename(columns={'sale_date': 'ds', 'quantity': 'y'})

# since multiple products can be sold on the same day, we group by date to get the total daily demand
prophet_df = prophet_df.groupby('ds')['y'].sum().reset_index()

# initializing the prophet model
# we can add built-in country holidays to make it smarter 
prophet_model = Prophet()
prophet_model.add_country_holidays(country_name='IN') # 'IN' stands for India

# training the prophet model on our daily data
prophet_model.fit(prophet_df)

# printing a success message
print("prophet model training completed successfully!")

Importing plotly failed. Interactive plots will not work.
D:\OFFICE\Tasks\sports_ai_system\sport_venv\lib\site-packages\holidays\countries\india.py:190: Warning: Requested Holidays are available only from 2001 to 2035.
  warnings.warn(warning_msg, Warning)
23:10:45 - cmdstanpy - INFO - Chain [1] start processing
23:10:46 - cmdstanpy - INFO - Chain [1] done processing


prophet model training completed successfully!


# evaluating the model predictions

In [22]:
# evaluating the random forest model 

# making predictions on our unseen test data
rf_predictions = rf_model.predict(x_test)

# calculating the error metrics for random forest
rf_mae = mean_absolute_error(y_test, rf_predictions)
rf_mse = mean_squared_error(y_test, rf_predictions)

print("--- random forest evaluation ---")
# mae tells us the average number of units we were off by
print(f"mean absolute error (mae): {rf_mae:.2f}")
print(f"mean squared error (mse): {rf_mse:.2f}")

--- random forest evaluation ---
mean absolute error (mae): 0.74
mean squared error (mse): 1.53


In [23]:
# evaluating the prophet model 

# prophet makes predictions on a dataframe that has a 'ds' (date) column
# we will predict on our historical dates to see how well it learned the patterns
prophet_predictions = prophet_model.predict(prophet_df[['ds']])

# calculating the error metrics for prophet
# prophet stores its predictions in a column called 'yhat'
prophet_mae = mean_absolute_error(prophet_df['y'], prophet_predictions['yhat'])
prophet_mse = mean_squared_error(prophet_df['y'], prophet_predictions['yhat'])

print("\n--- prophet evaluation ---")
print(f"mean absolute error (mae): {prophet_mae:.2f}")
print(f"mean squared error (mse): {prophet_mse:.2f}")


--- prophet evaluation ---
mean absolute error (mae): 1.09
mean squared error (mse): 1.73


# saving the trained model for the streamlit app

In [24]:
# we need the os library to make sure our models folder exists


# based on the evaluation, random forest is the winner
# ensuring the models directory exists before we try to save
os.makedirs('../models', exist_ok=True)

# defining the path where the model will be saved based on our folder structure
model_path = '../models/demand_model.pkl'

# saving the trained random forest model using joblib
joblib.dump(rf_model, model_path)

# printing a confirmation message so we know it worked
print(f"success! the best model has been saved to: {model_path}")

success! the best model has been saved to: ../models/demand_model.pkl
